In [1]:
with open("/mnt/d/forCoding_code/_init_tools/x__params.py", "r") as f:
    txt = f.read()
exec(txt)

with open("/mnt/d/forCoding_code/_init_tools/report_macro_0516.py", "r") as f:
    txt = f.read()
exec(txt)

with open("/mnt/d/forCoding_code/_init_tools/x__tool_here1.py", "r") as f:
    exec(f.read())
with open("/mnt/d/forCoding_code/_init_tools/x__tool_here2.py", "r") as f:
    exec(f.read())

wasted_dir = "./wasted"
if not os.path.exists(wasted_dir):
    os.makedirs(wasted_dir)

pd.set_option('display.max_rows',200)
pd.set_option('display.max_columns',200)

storage dir: /mnt/d/forCoding_data/QuantFinance/plan_3-standardization_1
code dir: /mnt/d/forCoding_code/QuantFinance/plan_3-standardization_1


In [2]:
!python daily_trading_system.py --clear-positions # --factor-progress

🚀 启动每日交易系统 - 2026-05-07 23:03
📊 开始更新每日数据...
🔐 登录baostock...
login success!
baostock登录成功
📅 更新日线数据...
🚀 开始高效并行增量更新日线数据...
🔧 需要更新 2998 只股票的数据
📝 需要更新的股票示例 (前10只):
   1. sh.600000 - 2026-04-28 到 2026-05-07
   2. sh.600004 - 2026-04-28 到 2026-05-07
   3. sh.600006 - 2026-04-28 到 2026-05-07
   4. sh.600007 - 2026-04-28 到 2026-05-07
   5. sh.600008 - 2026-04-28 到 2026-05-07
   6. sh.600009 - 2026-04-28 到 2026-05-07
   7. sh.600010 - 2026-04-28 到 2026-05-07
   8. sh.600011 - 2026-04-28 到 2026-05-07
   9. sh.600012 - 2026-04-28 到 2026-05-07
   10. sh.600015 - 2026-04-28 到 2026-05-07
日线并行下载:   0%|  | 1/2998 [00:01<1:11:27,  1.43s/股, 成功=1, 失败=0, 数据=5]login success!
login success!
login success!
login success!
日线并行下载:   0%|   | 2/2998 [00:02<50:02,  1.00s/股, 成功=2, 失败=0, 数据=10]login success!
日线并行下载: 100%|█| 2998/2998 [07:18<00:00,  6.84股/s, 成功=2998, 失败=0, 数据
✅ 日线更新完成: 成功 2998, 失败 0, 新增数据 15011 条
✅ 日线数据更新成功: 15011 条记录
📌 周线目标日期: 2026-05-01 | 待补全股票数: 5203
📈 更新周线数据...
🚀 开始高效并行增量更新周线数据...
🔧 需要更新 2998 只股票

In [ ]:
# import baostock as bs

# lg = bs.login()
# if lg.error_code != "0":
#     raise RuntimeError(f"baostock 登录失败: {lg.error_code} {lg.error_msg}")

# print("✅ baostock 登录成功")

# try:
#     rs = bs.query_history_k_data_plus(
#         "sh.000001",
#         "date,code,close",
#         start_date="2025-01-01",
#         end_date="2025-01-15",
#         frequency="d",
#         adjustflag="2",
#     )
#     if rs.error_code != "0":
#         raise RuntimeError(f"baostock 查询失败: {rs.error_code} {rs.error_msg}")
#     rows = 0
#     while rs.next():
#         rows += 1
#     if rows == 0:
#         print("⚠️ 查询成功但无返回数据")
#     else:
#         print(f"✅ 查询成功，返回 {rows} 行数据")
# finally:
#     bs.logout()
#     print("✅ baostock 已登出")


In [ ]:
## 写一个代码，检查一下数据库里面的数据，日线是不是已经更新到昨天的了，
## 以及周线是否已经更新到上周五了。

In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime, timedelta

DB_PATH = "/mnt/d/forCoding_data/QuantFinance/plan_3-standardization_1/stock_data.db"

def prev_trading_day(d):
    while d.weekday() >= 5:
        d -= timedelta(days=1)
    return d

today = datetime.now().date()
expected_daily = prev_trading_day(today - timedelta(days=1))

days_to_friday = (expected_daily.weekday() - 4) % 7
expected_weekly = expected_daily - timedelta(days=days_to_friday)

print(f"DB: {DB_PATH}")
print(f"期望(日线到昨天交易日): {expected_daily}")
print(f"期望(周线到上周五): {expected_weekly}")

conn = sqlite3.connect(DB_PATH)
try:
    daily_max = pd.read_sql_query("SELECT MAX(date) AS max_date FROM stock_daily", conn)["max_date"].iloc[0]
    weekly_max = pd.read_sql_query("SELECT MAX(date) AS max_date FROM stock_weekly", conn)["max_date"].iloc[0]

    print("-" * 60)
    print(f"全库日线最大日期: {daily_max}")
    print(f"全库周线最大日期: {weekly_max}")

    daily_ok = daily_max is not None and pd.to_datetime(daily_max).date() >= expected_daily
    weekly_ok = weekly_max is not None and pd.to_datetime(weekly_max).date() >= expected_weekly

    print("-" * 60)
    print(f"日线是否已到昨天交易日: {'✅' if daily_ok else '❌'}")
    print(f"周线是否已到上周五: {'✅' if weekly_ok else '❌'}")

    daily_by_stock = pd.read_sql_query(
        "SELECT stock_code, MAX(date) AS max_date FROM stock_daily GROUP BY stock_code",
        conn,
    )
    daily_by_stock["max_date_dt"] = pd.to_datetime(daily_by_stock["max_date"], errors="coerce")
    daily_lag = daily_by_stock[daily_by_stock["max_date_dt"].dt.date < expected_daily]

    weekly_by_stock = pd.read_sql_query(
        "SELECT stock_code, MAX(date) AS max_date FROM stock_weekly GROUP BY stock_code",
        conn,
    )
    weekly_by_stock["max_date_dt"] = pd.to_datetime(weekly_by_stock["max_date"], errors="coerce")
    weekly_lag = weekly_by_stock[weekly_by_stock["max_date_dt"].dt.date < expected_weekly]

    print("-" * 60)
    print(f"日线落后股票数: {len(daily_lag)} / {len(daily_by_stock)}")
    if len(daily_lag) > 0:
        print(daily_lag.sort_values("max_date").head(10)[["stock_code", "max_date"]])

    print("-" * 60)
    print(f"周线落后股票数: {len(weekly_lag)} / {len(weekly_by_stock)}")
    if len(weekly_lag) > 0:
        print(weekly_lag.sort_values("max_date").head(10)[["stock_code", "max_date"]])
finally:
    conn.close()


In [ ]:
# !pip install --upgrade baostock

In [ ]:
# import baostock as bs

# # 示例代码查看IP是否加入黑名单：          
# lg = bs.login()

# # 显示登陆返回信息

# print('login respond error_code:'+lg.error_code)

# print('login respond  error_msg:'+lg.error_msg)

# if(lg.error_code == "10001011"):

#     print("IP已经加入黑名单, 需要去QQ群里求助")

In [ ]:
# ip:
# 183.194.151.110

In [ ]:
!head /mnt/d/forCoding_data/QuantFinance/plan_3-standardization_1/recommendations/trading_recommendations_20260422.csv


In [ ]:
# baostock_connection_check.py
import sys

def main():
    try:
        import baostock as bs
    except Exception as e:
        print(f"❌ 导入 baostock 失败: {e}")
        sys.exit(2)

    lg = bs.login()
    if lg.error_code != "0":
        print(f"❌ baostock 登录失败: {lg.error_code} {lg.error_msg}")
        sys.exit(1)

    print("✅ baostock 登录成功")

    try:
        rs = bs.query_history_k_data_plus(
            "sh.000001",
            "date,code,close",
            start_date="2025-01-01",
            end_date="2025-01-15",
            frequency="d",
            adjustflag="2",
        )

        if rs.error_code != "0":
            print(f"❌ 查询失败: {rs.error_code} {rs.error_msg}")
            sys.exit(1)

        rows = 0
        while rs.next():
            rows += 1

        if rows == 0:
            print("⚠️ 查询成功但无返回数据（可能是日期区间无交易日/接口异常）")
        else:
            print(f"✅ 查询成功，返回 {rows} 行数据")

    finally:
        bs.logout()
        print("✅ baostock 已登出")

if __name__ == "__main__":
    main()

In [ ]:
!python run_weekly_update.py # --test

In [ ]:
!python calculate_stock_features.py

* 基于周线，在过去的2/4/8周，有过一阳穿四线，也就是一个上涨k线从下到上超越了5、10、20、30日均线。

* 周收盘价的原始数值。

* 是否ST股。

* 最近一周的日线里面是否有过涨停。

* 最近一周日线里是否有过涨停回调。

* 75周的周期内放量突破20周线次数大于等于2次。

* 5、10、20周均线发散走多。

* 最近一次突破20周均线后若有回调成交量缩量，且不跌破突破周的收盘价。

* 上市日期是否大于240天。

In [ ]:
df_rst = pd.read_csv("/mnt/d/forCoding_data/QuantFinance/plan_3-standardization_1/stock_features.csv")

In [ ]:
df_rst[
    (
        df_rst.listing_gt_240==1
    ) & (
        ~df_rst.is_st
    ) & (
        df_rst.cross_ma_4w==1
    ) & (
        df_rst.close_price<=30
    ) & (
        df_rst.volume_break_ge_2==1
    )
    #  & (
    #     df_rst.has_limit_up_pullback==1
    # )
]

In [ ]:
!python calculate_stock_features.py --test --stock sh.600000 

In [ ]:
from sql_query_tool import get_table_info
table_info = get_table_info('stocks')

In [ ]:
table_info

In [ ]:
from sql_query_tool import get_table_names  
tables = get_table_names()

In [ ]:
tables

In [ ]:
from sql_query_tool import query_with_params
# 正确示例（修复后）：
result = query_with_params(
    "SELECT * FROM stocks WHERE market = ? AND industry = ?", 
    ('SH', '银行')
)

In [ ]:
# 正确示例（修复后）：
result = query_with_params(
    "SELECT * FROM stocks", 
    ('')
)

In [ ]:
result[result.name.str.startswith("ST")]

# Recycle Bin

In [ ]:
!python run_weekly_update.py 

In [ ]:
!python debug_incremental_check.py --stock sh.600004

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
测试baostock API连接和单只股票数据获取
用于检查网络连接和API可用性
"""

import baostock as bs
import pandas as pd
import time
from datetime import datetime

def test_bao_connection():
    """测试baostock连接和单只股票数据获取"""
    
    print("🔍 开始测试baostock API连接...")
    print("-" * 50)
    
    # 1. 登录测试
    print("1. 正在登录baostock系统...")
    try:
        lg = bs.login()
        if lg.error_code != '0':
            print(f"❌ 登录失败: {lg.error_msg}")
            return False
        print(f"✅ 登录成功: {lg.error_msg}")
    except Exception as e:
        print(f"❌ 登录异常: {e}")
        return False
    
    # 2. 获取单只股票测试数据
    print("\n2. 正在获取单只股票周线数据...")
    test_stock_code = "sh.600000"  # 浦发银行
    
    try:
        # 获取最近4周的周线数据
        end_date = datetime.now().strftime("%Y-%m-%d")
        start_date = (datetime.now() - pd.DateOffset(weeks=4)).strftime("%Y-%m-%d")
        
        print(f"   股票代码: {test_stock_code}")
        print(f"   时间范围: {start_date} 至 {end_date}")
        
        rs = bs.query_history_k_data_plus(
            test_stock_code,
            "date,code,open,high,low,close,volume,amount,turn",
            start_date=start_date,
            end_date=end_date,
            frequency="w",  # 周线
            adjustflag="2"   # 前复权
        )
        
        if rs.error_code != '0':
            print(f"❌ 数据获取失败: {rs.error_msg}")
            bs.logout()
            return False
        
        # 解析数据
        data_list = []
        while (rs.error_code == '0') & rs.next():
            data_list.append(rs.get_row_data())
        
        if not data_list:
            print("❌ 未获取到数据")
            bs.logout()
            return False
        
        df = pd.DataFrame(data_list, columns=rs.fields)
        
        print(f"✅ 成功获取 {len(df)} 条周线数据")
        print("\n📊 最新数据预览:")
        print("-" * 80)
        print(df.tail().to_string(index=False))
        print("-" * 80)
        
        # 显示基本统计信息
        print("\n📈 数据统计:")
        print(f"   时间范围: {df['date'].min()} 至 {df['date'].max()}")
        print(f"   数据条数: {len(df)} 条")
        print(f"   最新收盘价: {df['close'].iloc[-1]} 元")
        
    except Exception as e:
        print(f"❌ 数据获取异常: {e}")
        bs.logout()
        return False
    
    # 3. 登出系统
    print("\n3. 正在登出系统...")
    try:
        bs.logout()
        print("✅ 登出成功")
    except Exception as e:
        print(f"⚠️  登出异常: {e}")
    
    print("\n" + "="*50)
    print("🎉 baostock API连接测试完成！")
    print("✅ 网络连接正常")
    print("✅ API接口可用")
    print("✅ 数据获取成功")
    print("="*50)
    
    return True

def test_multiple_stocks():
    """测试多只股票获取"""
    print("\n🔍 额外测试：多只股票快速连接测试...")
    
    test_codes = ["sh.600000", "sz.000001", "sh.601318"]  # 浦发银行、平安银行、中国平安
    
    for code in test_codes:
        try:
            rs = bs.query_history_k_data_plus(
                code,
                "date,close",
                start_date="2025-01-01",
                end_date=datetime.now().strftime("%Y-%m-%d"),
                frequency="w",
                adjustflag="2"
            )
            
            if rs.error_code == '0':
                print(f"   ✅ {code}: 连接成功")
            else:
                print(f"   ⚠️  {code}: {rs.error_msg}")
                
        except Exception as e:
            print(f"   ❌ {code}: 连接异常 - {e}")

if __name__ == "__main__":
    print("🐂 baostock API连接测试工具")
    print("📡 测试网络连接和数据获取能力")
    print()
    
    success = test_bao_connection()
    
    if success:
        # 如果主要测试成功，再进行多股票测试
        test_multiple_stocks()
    else:
        print("\n❌ 主要测试失败，请检查:")
        print("   1. 网络连接是否正常")
        print("   2. baostock服务是否可用")
        print("   3. 防火墙设置")
        print("   4. Python环境依赖")
    
    print("\n💡 提示: 如果测试失败，请检查:")
    print("   - 网络连接: ping www.baidu.com")
    print("   - 安装依赖: pip install baostock pandas")
    print("   - 服务状态: 访问baostock官网查看服务状态")

In [ ]:
# !ping www.baidu.com

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
修复版的baostock API连接测试
解决多只股票测试时的连接问题
"""

import baostock as bs
import pandas as pd
import time
from datetime import datetime

def test_single_stock(stock_code, start_date, end_date):
    """测试单只股票数据获取"""
    try:
        rs = bs.query_history_k_data_plus(
            stock_code,
            "date,code,open,high,low,close,volume,amount,turn",
            start_date=start_date,
            end_date=end_date,
            frequency="w",
            adjustflag="2"
        )
        
        if rs.error_code != '0':
            return False, f"API错误: {rs.error_msg}"
        
        data_list = []
        while (rs.error_code == '0') & rs.next():
            data_list.append(rs.get_row_data())
        
        if not data_list:
            return False, "无数据"
        
        df = pd.DataFrame(data_list, columns=rs.fields)
        return True, df
        
    except Exception as e:
        return False, f"异常: {str(e)}"

def test_bao_connection_comprehensive():
    """全面的baostock连接测试"""
    
    print("🔍 开始全面测试baostock API连接...")
    print("-" * 60)
    
    # 测试股票列表
    test_stocks = [
        {"code": "sh.600000", "name": "浦发银行"},
        {"code": "sz.000001", "name": "平安银行"}, 
        {"code": "sh.601318", "name": "中国平安"},
        {"code": "sh.600036", "name": "招商银行"},
        {"code": "sz.000858", "name": "五粮液"}
    ]
    
    success_count = 0
    total_tests = len(test_stocks)
    
    for i, stock in enumerate(test_stocks, 1):
        print(f"\n{i}/{total_tests} 测试 {stock['name']} ({stock['code']})...")
        
        try:
            # 每次测试都重新建立连接
            lg = bs.login()
            if lg.error_code != '0':
                print(f"   ❌ 登录失败: {lg.error_msg}")
                continue
            
            # 获取测试数据
            end_date = datetime.now().strftime("%Y-%m-%d")
            start_date = "2025-01-01"  # 固定开始日期
            
            success, result = test_single_stock(stock['code'], start_date, end_date)
            
            if success:
                df = result
                print(f"   ✅ 成功获取 {len(df)} 条数据")
                print(f"      最新收盘价: {df['close'].iloc[-1]}元, 日期: {df['date'].iloc[-1]}")
                success_count += 1
            else:
                print(f"   ❌ 失败: {result}")
            
            # 立即登出
            bs.logout()
            time.sleep(0.5)  # 短暂延迟避免频繁请求
            
        except Exception as e:
            print(f"   ❌ 测试异常: {e}")
    
    print("\n" + "="*60)
    print(f"📊 测试结果: {success_count}/{total_tests} 只股票测试成功")
    
    if success_count == total_tests:
        print("🎉 所有股票测试成功！网络连接和API接口正常")
    elif success_count > 0:
        print("⚠️  部分股票测试成功，可能存在个别股票数据问题")
    else:
        print("❌ 所有股票测试失败，请检查网络连接和baostock服务状态")
    
    print("="*60)
    return success_count

def quick_connection_test():
    """快速连接测试"""
    print("🚀 快速连接测试...")
    
    try:
        lg = bs.login()
        if lg.error_code != '0':
            print(f"❌ 登录失败: {lg.error_msg}")
            return False
        
        print("✅ 登录成功")
        
        # 简单查询测试
        rs = bs.query_stock_basic()
        if rs.error_code == '0':
            print("✅ 基础查询正常")
        else:
            print(f"❌ 基础查询失败: {rs.error_msg}")
        
        bs.logout()
        print("✅ 登出成功")
        return True
        
    except Exception as e:
        print(f"❌ 连接测试异常: {e}")
        return False

if __name__ == "__main__":
    print("🐂 baostock API连接测试工具 (修复版)")
    print("📡 解决多连接测试问题")
    print()
    
    # 先进行快速连接测试
    if not quick_connection_test():
        print("\n❌ 快速连接测试失败，无需继续完整测试")
        exit(1)
    
    print("\n" + "="*60)
    print("开始完整股票数据测试...")
    print("="*60)
    
    # 进行完整测试
    success_count = test_bao_connection_comprehensive()
    
    print("\n💡 故障排查建议:")
    if success_count == 0:
        print("1. 🔌 检查网络连接: ping www.baidu.com")
        print("2. 🌐 检查baostock服务状态")
        print("3. 🔧 重新安装依赖: pip install --upgrade baostock")
        print("4. 🕐 避免频繁请求，添加适当延迟")
    else:
        print("✅ 连接正常，可以开始数据下载")

In [ ]:
!python test_parallel_performance.py